# Detecção de Anomalias

Neste notebook, iremos explorar o modelo **EfficientAD**, disponibilizado pela biblioteca `anomalib`, para detectar anomalias (defeitos) em imagens.

Diferente dos modelos de classificação e detecção que vimos até agora, um modelo de detecção de anomalias é treinado **apenas com imagens normais** (sem defeito). Durante a inferência, ele aprende o que é "normal" e sinaliza como anomalia tudo que se desvia muito disso, sem nunca ter visto um exemplo de defeito durante o treino.

Aqui vamos aproveitar um modelo já treinado (checkpoint `modelos/efficientad-bottle-epoch=19.ckpt`) na categoria `bottle` do dataset MVTec AD, e usá-lo apenas para fazer previsões em imagens novas.

> Caso queira treinar um novo modelo, utilize os scripts encontrados em `scripts/04/` na seguinte ordem:
> 1. `download_imagenette.py`: para baixar o dataset **Imagenette**, necessário para o **EfficientAD**
> 2. `download_mvtecad.py`: para baixar o dataset **MCTec AD**, que contém as imagens da categoria `bottle`.
> 3. `train_efficientad.py`: para treinar o modelo. 

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install anomalib==2.5.1
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que já usamos em outros notebooks, vamos precisar de mais algumas peças da `anomalib`:
* `EfficientAd`: a arquitetura do modelo.
* `Engine`: orquestra o treino e a inferência (é quem sabe carregar um checkpoint e rodar o modelo nas imagens).
* `PredictDataset`: monta um dataset a partir de uma imagem ou de uma pasta de imagens, sem precisar de rótulos.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path

from anomalib.data import PredictDataset
from anomalib.models import EfficientAd
from anomalib.engine import Engine

import torch

import numpy as np
import matplotlib.pyplot as plt
# Indica ao notebook to render figures in-page.
%matplotlib inline

## 1. Carregando o modelo a partir do checkpoint

O checkpoint `modelos/efficientad-bottle-epoch=19.ckpt` já contém os pesos de um `EfficientAd` treinado por 20 épocas apenas com imagens normais ("good") da categoria `bottle`.

Não precisamos treinar nada agora: basta criar uma instância do modelo e do `Engine`, e indicar o caminho do checkpoint no momento da previsão.

In [ ]:
model = EfficientAd()

engine = Engine(
    accelerator="cpu",
    devices=1,
    enable_progress_bar=False,
    logger=False,
    default_root_dir="output/results"
)

checkpoint_path = "modelos/efficientad-bottle-epoch=19.ckpt"

## 2. Fazendo previsão em uma imagem

O método `engine.predict` aceita um dataset (`dataset`) e roda o modelo em todas as imagens que possui, carregando os pesos indicados em `ckpt_path`.

In [ ]:
dataset = PredictDataset(path="imagens/04/bottle_test/broken_large/000.png")

predictions = engine.predict(
    model=model, 
    dataset=dataset, 
    ckpt_path=checkpoint_path
)

### 2.1. Interpretando a saída

Podemos usar as propriedades `pred_score` e `pred_label` para identificar se a imagem é uma anomalia. 

- `pred_score`: valores acima de `0.5` indicam anomalia.
- `pred_label`: `True` quando anomalia.

In [ ]:
prediction = predictions[0]

print("Score:", prediction.pred_score)
print("Label", prediction.pred_label)

A propriedade `anomaly_map` traz o mapa de calor da anomalia.

In [ ]:
heatmap = prediction.anomaly_map.cpu().permute(1, 2, 0).numpy()
heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())

plt.figure(figsize=(6,6))
plt.imshow(heatmap, cmap="jet")
plt.colorbar()
plt.show()

A propriedade `pred_mask` traz a máscara binária de onde se encontra as anomalias na imagem.

In [ ]:
mask = prediction.pred_mask.cpu().permute(1, 2, 0).numpy()

plt.figure(figsize=(6,6))
plt.imshow(mask, cmap="gray")
plt.show()

## 3. Fazendo previsões em imagens novas

O dataset MVTec AD organiza as imagens de teste da categoria `bottle` em subpastas, cada uma representando uma situação diferente:
* `good`: garrafas normais, sem defeito.
* `broken_large` / `broken_small`: garrafas com quebras grandes/pequenas.
* `contamination`: garrafas com contaminação no líquido.

O método `engine.predict` aceita o caminho de uma pasta (`data_path`) e roda o modelo em todas as imagens dela, carregando os pesos indicados em `ckpt_path`.

In [ ]:
previsoes_boas = engine.predict(
    model=model,
    data_path="imagens/04/bottle_test/good",
    ckpt_path=checkpoint_path,
)

Cada chamada de `predict` devolve uma lista de `ImageBatch` — como não configuramos um tamanho de lote maior, cada `ImageBatch` da lista contém uma única imagem. Cada `ImageBatch` pode ser percorrido item a item (`.items`), e cada item traz:
* `image` / `image_path`: a imagem de entrada.
* `pred_score`: a pontuação de anomalia prevista (quanto maior, mais anômala).
* `pred_label`: `True` se o modelo classificou a imagem como anomalia.
* `anomaly_map`: o mapa de calor de anomalia, pixel a pixel.
* `pred_mask`: a máscara binária da região apontada como defeito.

Para facilitar, vamos "desempacotar" a lista de lotes em uma lista simples de itens.

In [ ]:
itens_boas = [item for lote in previsoes_boas for item in lote.items]

for item in itens_boas:
    nome = Path(item.image_path).name
    status = "ANOMALIA" if item.pred_label else "NORMAL"
    print(f"{nome}: {status} (score={float(item.pred_score):.3f})")

### 3.1. Visualizando os resultados

Vamos criar a função `apresentar_previsao`, para montar lado a lado: a imagem original, o mapa de anomalia sobreposto e a máscara prevista sobreposta.

In [ ]:
def apresentar_previsao(item) -> None:

    status = "ANOMALIA" if item.pred_label else "NORMAL"

    # ------------------
    # Imagem
    # ------------------
    img = item.image

    if torch.is_tensor(img):
        img = img.detach().cpu().permute(1, 2, 0).numpy()

    img = np.clip(img, 0.0, 1.0)

    # ------------------
    # Anomaly Map
    # ------------------
    anomaly_map = item.anomaly_map

    if torch.is_tensor(anomaly_map):
        anomaly_map = anomaly_map.detach().cpu().numpy()

    anomaly_map_norm = (anomaly_map - anomaly_map.min()) / (anomaly_map.max() - anomaly_map.min() + 1e-8)

    # ------------------
    # Máscara do objeto
    # ------------------
    gray = img.mean(axis=2)

    # Fundo branco
    obj_mask = gray < 0.95

    # Aplicar máscara
    anomaly_map_masked = anomaly_map_norm.copy()
    anomaly_map_masked[~obj_mask] = np.nan

    # ------------------
    # Destacar apenas regiões mais relevantes
    # ------------------
    threshold = np.percentile(
        anomaly_map_masked[~np.isnan(anomaly_map_masked)],
        90
    )

    overlay_map = anomaly_map_masked.copy()
    overlay_map[overlay_map < threshold] = np.nan

    # ------------------
    # Pred Mask
    # ------------------
    pred_mask = item.pred_mask

    if torch.is_tensor(pred_mask):
        pred_mask = pred_mask.detach().cpu().numpy()

    # ------------------
    # Plot
    # ------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Imagem
    axes[0].imshow(img)
    axes[0].set_title("Imagem")
    axes[0].axis("off")

    # Heatmap
    axes[1].imshow(
        anomaly_map_masked,
        cmap="jet"
    )

    axes[1].set_title("Mapa de Calor")
    axes[1].axis("off")

    # Máscara prevista
    axes[2].imshow(img)

    if pred_mask is not None:
        axes[2].contour(
            pred_mask,
            levels=[0.5],
            colors="red",
            linewidths=2
        )

    axes[2].set_title("Máscara")
    axes[2].axis("off")

    plt.suptitle(
        f"{Path(item.image_path).name} | {status} | score={float(item.pred_score):.3f}",
        fontsize=18,
        fontweight="bold"
    )

    plt.show()

In [ ]:
for item in itens_boas[:3]:
    apresentar_previsao(item)

## 4. Comparando com imagens com defeito

Agora vamos repetir o processo para as pastas com garrafas defeituosas. O esperado é que o modelo aponte `pred_label=True` e que o mapa de anomalia destaque justamente a região do defeito.

In [ ]:
previsoes_quebradas = engine.predict(
    model=model,
    data_path="imagens/04/bottle_test/broken_large",
    ckpt_path=checkpoint_path,
)
itens_quebradas = [item for lote in previsoes_quebradas for item in lote.items]

for item in itens_quebradas[:3]:
    apresentar_previsao(item)

In [ ]:
previsoes_contaminacao = engine.predict(
    model=model,
    data_path="imagens/04/bottle_test/contamination",
    ckpt_path=checkpoint_path,
)
itens_contaminacao = [item for lote in previsoes_contaminacao for item in lote.items]

for item in itens_contaminacao[:3]:
    apresentar_previsao(item)

## 5. Resumindo os resultados

Por fim, vamos conferir de forma resumida quantas imagens de cada pasta o modelo classificou como anomalia.

In [ ]:
conjuntos = {
    "good": itens_boas,
    "broken_large": itens_quebradas,
    "contamination": itens_contaminacao,
}

for nome, itens in conjuntos.items():
    anomalias = sum(bool(item.pred_label) for item in itens)
    print(f"{nome}: {anomalias}/{len(itens)} imagens classificadas como anomalia")